<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/01_dataset_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part-01: Environment Setup for Data-Analysis

In [ ]:
# DATA ANALYSIS NOTEBOOK SETUP

from pathlib import Path

# CONFIG
REPO_URL = "https://github.com/Al-Amin95/PromptInjectionDetectionSystem.git"
BRANCH = "data-analysis"
REPO_DIR = Path("/content/PromptInjectionDetectionSystem")

# CLONE OR UPDATE REPO
if REPO_DIR.exists():
    print("Repo exists → updating...")
    %cd /content/PromptInjectionDetectionSystem
    !git checkout {BRANCH}
    !git pull
else:
    print("Cloning repo...")
    %cd /content
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/PromptInjectionDetectionSystem

# INSTALL REQUIREMENTS
print("Installing dependencies...")
!pip install -q -r requirements.txt

# DEFINE PATHS
PROJECT_ROOT = Path("/content/PromptInjectionDetectionSystem")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Dataset files
TRAIN_PATH = RAW_DATA_DIR / "train-00000-of-00001-9564e8b05b4757ab.parquet"
TEST_PATH  = RAW_DATA_DIR / "test-00000-of-00001-701d16158af87368.parquet"

# VERIFY FILES
print("\nChecking dataset files...")
print("Train exists:", TRAIN_PATH.exists())
print("Test exists :", TEST_PATH.exists())

# QUICK LOAD CHECK
import pandas as pd

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("\nDataset loaded successfully.")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

#Part-02: Dataset Loading & Validation

In [ ]:
# 1-Confirm train and test dataframes exist

required_dataframes = ["train_df", "test_df"]

for dataframe_name in required_dataframes:
    if dataframe_name not in globals():
        raise NameError(f"{dataframe_name} is missing. Run the setup cell first.")

print("Required dataframes found.")
print("train_df type:", type(train_df))
print("test_df type:", type(test_df))

In [ ]:
# 2-Verify train and test dataset shapes

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

assert train_df.shape == (546, 2), "Unexpected train_df shape."
assert test_df.shape == (116, 2), "Unexpected test_df shape."

print("Train and test dataset shapes are correct.")

In [ ]:
# 3-Verify column structure

expected_columns = ["text", "label"]

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

assert train_df.columns.tolist() == expected_columns, "Train columns are incorrect."
assert test_df.columns.tolist() == expected_columns, "Test columns are incorrect."

print("Column structure is correct.")

In [ ]:
# 4-Verify label values and define label meaning

label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

train_unique_labels = sorted(train_df["label"].unique().tolist())
test_unique_labels = sorted(test_df["label"].unique().tolist())

print("Train unique labels:", train_unique_labels)
print("Test unique labels:", test_unique_labels)
print("Label mapping:", label_mapping)

assert train_unique_labels == [0, 1], "Train labels are not valid binary labels."
assert test_unique_labels == [0, 1], "Test labels are not valid binary labels."

print("Label values and label meaning are correct.")

In [ ]:
# 5-Preview raw train and test samples

print("First 5 rows from train_df:")
display(train_df.head())
print("\n")

print("First 5 rows from test_df:")
display(test_df.head())

#Part-03: Dataset Merge


In [ ]:
# 1-Create temporary train and test copies with split labels

train_eda_df = train_df.copy()
test_eda_df = test_df.copy()

train_eda_df["split"] = "train"
test_eda_df["split"] = "test"

print("Temporary train EDA dataset created.")
print("Temporary test EDA dataset created.")

print("train_eda_df shape:", train_eda_df.shape)
print("test_eda_df shape:", test_eda_df.shape)

print("\ntrain_eda_df columns:", train_eda_df.columns.tolist())
print("test_eda_df columns:", test_eda_df.columns.tolist())

In [ ]:
# 2-Combine train and test data for exploratory analysis only

combined_eda_df = pd.concat(
    [train_eda_df, test_eda_df],
    axis=0,
    ignore_index=True
)

print("Combined EDA dataset created.")
print("combined_eda_df shape:", combined_eda_df.shape)
print("combined_eda_df columns:", combined_eda_df.columns.tolist())

In [ ]:
# 3-Verify split counts in combined EDA dataset

split_counts = combined_eda_df["split"].value_counts().sort_index()

print("Split counts:")
print(split_counts)

assert split_counts["train"] == 546, "Train split count is incorrect."
assert split_counts["test"] == 116, "Test split count is incorrect."

print("\nSplit counts are correct.")

In [ ]:
# 4-Confirm combined EDA dataset is analysis-only

print("Combined EDA dataset confirmation:")
print("- combined_eda_df is used only for exploratory data analysis.")
print("- It must not be used directly for model training.")
print("- The split column is only an analysis helper column.")

print("\nOriginal dataset shapes remain unchanged:")
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

#Part-04: Data Quality Validation


In [ ]:
# 1-Confirm required dataframes exist for data quality validation

required_dataframes = ["train_df", "test_df", "combined_eda_df"]

for dataframe_name in required_dataframes:
    if dataframe_name not in globals():
        raise NameError(f"{dataframe_name} is missing. Run previous parts first.")

print("Required dataframes found.")
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("combined_eda_df shape:", combined_eda_df.shape)

In [ ]:
# 2-Check missing values in text and label columns

missing_values_summary = pd.DataFrame([
    {
        "dataset": "train",
        "missing_text": train_df["text"].isna().sum(),
        "missing_label": train_df["label"].isna().sum()
    },
    {
        "dataset": "test",
        "missing_text": test_df["text"].isna().sum(),
        "missing_label": test_df["label"].isna().sum()
    },
    {
        "dataset": "combined",
        "missing_text": combined_eda_df["text"].isna().sum(),
        "missing_label": combined_eda_df["label"].isna().sum()
    }
])

print("Missing values summary:")
display(missing_values_summary)

In [ ]:
# 3-Check empty and whitespace-only text values

def count_empty_text(dataframe):
    return (dataframe["text"].astype(str) == "").sum()

def count_whitespace_only_text(dataframe):
    return dataframe["text"].astype(str).str.strip().eq("").sum()

empty_whitespace_summary = pd.DataFrame([
    {
        "dataset": "train",
        "empty_text": count_empty_text(train_df),
        "whitespace_only_text": count_whitespace_only_text(train_df)
    },
    {
        "dataset": "test",
        "empty_text": count_empty_text(test_df),
        "whitespace_only_text": count_whitespace_only_text(test_df)
    },
    {
        "dataset": "combined",
        "empty_text": count_empty_text(combined_eda_df),
        "whitespace_only_text": count_whitespace_only_text(combined_eda_df)
    }
])

print("Empty and whitespace-only text summary:")
display(empty_whitespace_summary)

In [ ]:
# 4-Check non-string text values

non_string_summary = pd.DataFrame([
    {
        "dataset": "train",
        "non_string_text": train_df["text"].apply(lambda value: not isinstance(value, str)).sum()
    },
    {
        "dataset": "test",
        "non_string_text": test_df["text"].apply(lambda value: not isinstance(value, str)).sum()
    },
    {
        "dataset": "combined",
        "non_string_text": combined_eda_df["text"].apply(lambda value: not isinstance(value, str)).sum()
    }
])

print("Non-string text summary:")
display(non_string_summary)

In [ ]:
# 5-Check punctuation-only or symbol-only text values

import re

def is_symbol_only_text(text):
    if not isinstance(text, str):
        return False

    stripped_text = text.strip()

    if stripped_text == "":
        return False

    return not any(character.isalnum() for character in stripped_text)

symbol_only_summary = pd.DataFrame([
    {
        "dataset": "train",
        "symbol_only_text": train_df["text"].apply(is_symbol_only_text).sum()
    },
    {
        "dataset": "test",
        "symbol_only_text": test_df["text"].apply(is_symbol_only_text).sum()
    },
    {
        "dataset": "combined",
        "symbol_only_text": combined_eda_df["text"].apply(is_symbol_only_text).sum()
    }
])

print("Punctuation-only or symbol-only text summary:")
display(symbol_only_summary)

In [ ]:
# 6-Create final data quality validation summary

data_quality_validation_summary = pd.DataFrame([
    {
        "check_name": "train_missing_text",
        "issue_count": train_df["text"].isna().sum(),
        "status": "Pass" if train_df["text"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "test_missing_text",
        "issue_count": test_df["text"].isna().sum(),
        "status": "Pass" if test_df["text"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "combined_missing_text",
        "issue_count": combined_eda_df["text"].isna().sum(),
        "status": "Pass" if combined_eda_df["text"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "train_missing_label",
        "issue_count": train_df["label"].isna().sum(),
        "status": "Pass" if train_df["label"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "test_missing_label",
        "issue_count": test_df["label"].isna().sum(),
        "status": "Pass" if test_df["label"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "combined_missing_label",
        "issue_count": combined_eda_df["label"].isna().sum(),
        "status": "Pass" if combined_eda_df["label"].isna().sum() == 0 else "Review"
    },
    {
        "check_name": "combined_empty_text",
        "issue_count": count_empty_text(combined_eda_df),
        "status": "Pass" if count_empty_text(combined_eda_df) == 0 else "Review"
    },
    {
        "check_name": "combined_whitespace_only_text",
        "issue_count": count_whitespace_only_text(combined_eda_df),
        "status": "Pass" if count_whitespace_only_text(combined_eda_df) == 0 else "Review"
    },
    {
        "check_name": "combined_non_string_text",
        "issue_count": combined_eda_df["text"].apply(lambda value: not isinstance(value, str)).sum(),
        "status": "Pass" if combined_eda_df["text"].apply(lambda value: not isinstance(value, str)).sum() == 0 else "Review"
    },
    {
        "check_name": "combined_symbol_only_text",
        "issue_count": combined_eda_df["text"].apply(is_symbol_only_text).sum(),
        "status": "Pass" if combined_eda_df["text"].apply(is_symbol_only_text).sum() == 0 else "Review"
    }
])

print("Final data quality validation summary:")
display(data_quality_validation_summary)

In [ ]:
# 7-Save data quality validation summary

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

data_quality_validation_path = TABLES_DIR / "missing_empty_invalid_text_report.csv"

data_quality_validation_summary.to_csv(data_quality_validation_path, index=False)

print("Data quality validation report saved to:", data_quality_validation_path)
print("File exists:", data_quality_validation_path.exists())

#Part-05: Duplicate & Leakage Detection

In [ ]:
# 1-Check exact duplicate full rows and duplicate text values

duplicate_summary_exact = pd.DataFrame([
    {
        "dataset": "train",
        "duplicate_full_rows": train_df.duplicated().sum(),
        "duplicate_text_rows": train_df["text"].duplicated(keep=False).sum()
    },
    {
        "dataset": "test",
        "duplicate_full_rows": test_df.duplicated().sum(),
        "duplicate_text_rows": test_df["text"].duplicated(keep=False).sum()
    },
    {
        "dataset": "combined",
        "duplicate_full_rows": combined_eda_df[["text", "label"]].duplicated().sum(),
        "duplicate_text_rows": combined_eda_df["text"].duplicated(keep=False).sum()
    }
])

print("Exact duplicate summary:")
display(duplicate_summary_exact)

In [ ]:
# 2-Check exact text values with conflicting labels

exact_label_counts_per_text = (
    combined_eda_df
    .groupby("text")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

exact_conflicting_texts = exact_label_counts_per_text[
    exact_label_counts_per_text["unique_label_count"] > 1
]

exact_conflicting_label_count = len(exact_conflicting_texts)

print("Exact conflicting text label count:", exact_conflicting_label_count)

if exact_conflicting_label_count > 0:
    display(exact_conflicting_texts.head())
else:
    print("No exact conflicting labels found.")

In [ ]:
# 3-Check exact train-test text overlap

train_text_set = set(train_df["text"])
test_text_set = set(test_df["text"])

exact_train_test_overlap = train_text_set.intersection(test_text_set)
exact_train_test_overlap_count = len(exact_train_test_overlap)

print("Exact train-test overlapping text count:", exact_train_test_overlap_count)

if exact_train_test_overlap_count > 0:
    exact_overlap_examples = combined_eda_df[
        combined_eda_df["text"].isin(exact_train_test_overlap)
    ].sort_values("text")
    display(exact_overlap_examples.head())
else:
    print("No exact train-test overlap found.")

In [ ]:
# 4-Create temporary normalised text for duplicate and leakage checks

def normalise_text_for_duplicate_check(text):
    text = str(text).lower().strip()
    text = " ".join(text.split())
    return text

duplicate_leakage_analysis_df = combined_eda_df.copy()

duplicate_leakage_analysis_df["normalised_text"] = duplicate_leakage_analysis_df["text"].apply(
    normalise_text_for_duplicate_check
)

print("Temporary normalised text created.")
print("duplicate_leakage_analysis_df shape:", duplicate_leakage_analysis_df.shape)
print("Columns:", duplicate_leakage_analysis_df.columns.tolist())

In [ ]:
# 5-Check normalised duplicates, conflicting labels, and train-test leakage

normalised_duplicate_text_rows = duplicate_leakage_analysis_df["normalised_text"].duplicated(
    keep=False
).sum()

normalised_label_counts_per_text = (
    duplicate_leakage_analysis_df
    .groupby("normalised_text")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

normalised_conflicting_texts = normalised_label_counts_per_text[
    normalised_label_counts_per_text["unique_label_count"] > 1
]

normalised_conflicting_label_count = len(normalised_conflicting_texts)

normalised_train_text_set = set(
    duplicate_leakage_analysis_df[
        duplicate_leakage_analysis_df["split"] == "train"
    ]["normalised_text"]
)

normalised_test_text_set = set(
    duplicate_leakage_analysis_df[
        duplicate_leakage_analysis_df["split"] == "test"
    ]["normalised_text"]
)

normalised_train_test_overlap = normalised_train_text_set.intersection(normalised_test_text_set)
normalised_train_test_overlap_count = len(normalised_train_test_overlap)

print("Normalised duplicate text rows:", normalised_duplicate_text_rows)
print("Normalised conflicting label count:", normalised_conflicting_label_count)
print("Normalised train-test overlapping text count:", normalised_train_test_overlap_count)

In [ ]:
# 6-Create and save duplicate/leakage summary

duplicate_leakage_summary = pd.DataFrame([
    {
        "check_name": "train_duplicate_full_rows",
        "issue_count": train_df.duplicated().sum(),
        "status": "Pass" if train_df.duplicated().sum() == 0 else "Review"
    },
    {
        "check_name": "test_duplicate_full_rows",
        "issue_count": test_df.duplicated().sum(),
        "status": "Pass" if test_df.duplicated().sum() == 0 else "Review"
    },
    {
        "check_name": "combined_duplicate_full_rows",
        "issue_count": combined_eda_df[["text", "label"]].duplicated().sum(),
        "status": "Pass" if combined_eda_df[["text", "label"]].duplicated().sum() == 0 else "Review"
    },
    {
        "check_name": "combined_duplicate_text_rows",
        "issue_count": combined_eda_df["text"].duplicated(keep=False).sum(),
        "status": "Pass" if combined_eda_df["text"].duplicated(keep=False).sum() == 0 else "Review"
    },
    {
        "check_name": "exact_conflicting_label_texts",
        "issue_count": exact_conflicting_label_count,
        "status": "Pass" if exact_conflicting_label_count == 0 else "Review"
    },
    {
        "check_name": "exact_train_test_overlap_texts",
        "issue_count": exact_train_test_overlap_count,
        "status": "Pass" if exact_train_test_overlap_count == 0 else "Review"
    },
    {
        "check_name": "normalised_duplicate_text_rows",
        "issue_count": normalised_duplicate_text_rows,
        "status": "Pass" if normalised_duplicate_text_rows == 0 else "Review"
    },
    {
        "check_name": "normalised_conflicting_label_texts",
        "issue_count": normalised_conflicting_label_count,
        "status": "Pass" if normalised_conflicting_label_count == 0 else "Review"
    },
    {
        "check_name": "normalised_train_test_overlap_texts",
        "issue_count": normalised_train_test_overlap_count,
        "status": "Pass" if normalised_train_test_overlap_count == 0 else "Review"
    }
])

print("Duplicate and leakage summary:")
display(duplicate_leakage_summary)

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

duplicate_leakage_summary_path = TABLES_DIR / "duplicate_leakage_summary.csv"
duplicate_leakage_summary.to_csv(duplicate_leakage_summary_path, index=False)

print("Duplicate/leakage summary saved to:", duplicate_leakage_summary_path)
print("File exists:", duplicate_leakage_summary_path.exists())

#Part-06: Class Distribution Analysis

In [ ]:
# 1-Count labels by split

label_distribution_by_split = (
    combined_eda_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

label_distribution_by_split["label_name"] = label_distribution_by_split["label"].map(label_mapping)

label_distribution_by_split = label_distribution_by_split[
    ["split", "label", "label_name", "count"]
].sort_values(["split", "label"]).reset_index(drop=True)

print("Label distribution by split:")
display(label_distribution_by_split)

In [ ]:
# 2-Calculate label percentages by split

label_distribution_by_split["percentage"] = (
    label_distribution_by_split
    .groupby("split")["count"]
    .transform(lambda values: (values / values.sum()) * 100)
    .round(2)
)

print("Label distribution by split with percentages:")
display(label_distribution_by_split)

In [ ]:
# 3-Create class balance comparison including combined dataset

class_balance_comparison = (
    combined_eda_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

combined_class_distribution = (
    combined_eda_df
    .groupby("label")
    .size()
    .reset_index(name="count")
)

combined_class_distribution["split"] = "combined"

class_balance_comparison = pd.concat(
    [
        class_balance_comparison,
        combined_class_distribution[["split", "label", "count"]]
    ],
    axis=0,
    ignore_index=True
)

class_balance_comparison["label_name"] = class_balance_comparison["label"].map(label_mapping)

class_balance_comparison["percentage"] = (
    class_balance_comparison
    .groupby("split")["count"]
    .transform(lambda values: (values / values.sum()) * 100)
    .round(2)
)

class_balance_comparison = class_balance_comparison[
    ["split", "label", "label_name", "count", "percentage"]
].sort_values(["split", "label"]).reset_index(drop=True)

print("Class balance comparison:")
display(class_balance_comparison)

In [ ]:
# 4-Save class distribution outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

label_distribution_path = TABLES_DIR / "label_distribution_by_split.csv"
class_balance_path = TABLES_DIR / "class_balance_comparison.csv"

label_distribution_by_split.to_csv(label_distribution_path, index=False)
class_balance_comparison.to_csv(class_balance_path, index=False)

print("Label distribution saved to:", label_distribution_path)
print("File exists:", label_distribution_path.exists())

print("\nClass balance comparison saved to:", class_balance_path)
print("File exists:", class_balance_path.exists())

#Part-07: Text Length Analysis

In [ ]:
# 1-Add character length and word count for analysis

combined_eda_df["character_length"] = combined_eda_df["text"].astype(str).str.len()

combined_eda_df["word_count"] = (
    combined_eda_df["text"]
    .astype(str)
    .str.split()
    .str.len()
)

print("Text length columns added.")
print("combined_eda_df shape:", combined_eda_df.shape)
print("Columns:", combined_eda_df.columns.tolist())

print("\nPreview:")
display(combined_eda_df[["text", "label", "split", "character_length", "word_count"]].head())

In [ ]:
# 2-Create character length summary by split and label

character_length_summary = (
    combined_eda_df
    .groupby(["split", "label"])
    .agg(
        row_count=("character_length", "count"),
        min_character_length=("character_length", "min"),
        median_character_length=("character_length", "median"),
        mean_character_length=("character_length", "mean"),
        max_character_length=("character_length", "max")
    )
    .reset_index()
)

character_length_summary["label_name"] = character_length_summary["label"].map(label_mapping)

character_length_summary = character_length_summary[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_character_length",
        "median_character_length",
        "mean_character_length",
        "max_character_length"
    ]
].sort_values(["split", "label"]).reset_index(drop=True)

character_length_summary["mean_character_length"] = character_length_summary["mean_character_length"].round(2)

print("Character length summary by split and label:")
display(character_length_summary)

In [ ]:
# 3-Create word count summary by split and label

word_count_summary = (
    combined_eda_df
    .groupby(["split", "label"])
    .agg(
        row_count=("word_count", "count"),
        min_word_count=("word_count", "min"),
        median_word_count=("word_count", "median"),
        mean_word_count=("word_count", "mean"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
)

word_count_summary["label_name"] = word_count_summary["label"].map(label_mapping)

word_count_summary = word_count_summary[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_word_count",
        "median_word_count",
        "mean_word_count",
        "max_word_count"
    ]
].sort_values(["split", "label"]).reset_index(drop=True)

word_count_summary["mean_word_count"] = word_count_summary["mean_word_count"].round(2)

print("Word count summary by split and label:")
display(word_count_summary)

In [ ]:
# 4-Create overall text length summary by label

overall_length_summary_by_label = (
    combined_eda_df
    .groupby("label")
    .agg(
        prompt_count=("text", "count"),
        min_character_length=("character_length", "min"),
        median_character_length=("character_length", "median"),
        mean_character_length=("character_length", "mean"),
        max_character_length=("character_length", "max"),
        min_word_count=("word_count", "min"),
        median_word_count=("word_count", "median"),
        mean_word_count=("word_count", "mean"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
)

overall_length_summary_by_label["label_name"] = overall_length_summary_by_label["label"].map(label_mapping)

overall_length_summary_by_label = overall_length_summary_by_label[
    [
        "label",
        "label_name",
        "prompt_count",
        "min_character_length",
        "median_character_length",
        "mean_character_length",
        "max_character_length",
        "min_word_count",
        "median_word_count",
        "mean_word_count",
        "max_word_count"
    ]
].sort_values("label").reset_index(drop=True)

overall_length_summary_by_label["mean_character_length"] = overall_length_summary_by_label["mean_character_length"].round(2)
overall_length_summary_by_label["mean_word_count"] = overall_length_summary_by_label["mean_word_count"].round(2)

print("Overall text length summary by label:")
display(overall_length_summary_by_label)

In [ ]:
# 5-Save text length analysis outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

text_length_analysis_path = TABLES_DIR / "text_length_analysis.csv"
character_length_summary_path = TABLES_DIR / "character_length_summary_by_split_label.csv"
word_count_summary_path = TABLES_DIR / "word_count_summary_by_split_label.csv"
overall_length_summary_path = TABLES_DIR / "overall_length_summary_by_label.csv"

combined_eda_df.to_csv(text_length_analysis_path, index=False)
character_length_summary.to_csv(character_length_summary_path, index=False)
word_count_summary.to_csv(word_count_summary_path, index=False)
overall_length_summary_by_label.to_csv(overall_length_summary_path, index=False)

print("Text length analysis saved to:", text_length_analysis_path)
print("File exists:", text_length_analysis_path.exists())

print("\nCharacter length summary saved to:", character_length_summary_path)
print("File exists:", character_length_summary_path.exists())

print("\nWord count summary saved to:", word_count_summary_path)
print("File exists:", word_count_summary_path.exists())

print("\nOverall length summary saved to:", overall_length_summary_path)
print("File exists:", overall_length_summary_path.exists())

#Part-08: Token Length Analysis

In [ ]:
# 1-Load tokenizers for token length analysis

from transformers import AutoTokenizer

DISTILBERT_CHECKPOINT = "distilbert-base-uncased"
ROBERTA_CHECKPOINT = "roberta-base"
SECBERT_CHECKPOINT = "jackaduma/SecBERT"

distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_CHECKPOINT)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CHECKPOINT)
secbert_tokenizer = AutoTokenizer.from_pretrained(SECBERT_CHECKPOINT)

print("Tokenizers loaded successfully.")

print("\nDistilBERT checkpoint:", DISTILBERT_CHECKPOINT)
print("DistilBERT tokenizer class:", distilbert_tokenizer.__class__.__name__)

print("\nRoBERTa checkpoint:", ROBERTA_CHECKPOINT)
print("RoBERTa tokenizer class:", roberta_tokenizer.__class__.__name__)

print("\nSecBERT checkpoint:", SECBERT_CHECKPOINT)
print("SecBERT tokenizer class:", secbert_tokenizer.__class__.__name__)

In [ ]:
# 2-Calculate token lengths for each tokenizer

def calculate_token_length(text, tokenizer):
    encoded_text = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    return len(encoded_text["input_ids"])

combined_eda_df["distilbert_token_length"] = combined_eda_df["text"].apply(
    lambda text: calculate_token_length(text, distilbert_tokenizer)
)

combined_eda_df["roberta_token_length"] = combined_eda_df["text"].apply(
    lambda text: calculate_token_length(text, roberta_tokenizer)
)

combined_eda_df["secbert_token_length"] = combined_eda_df["text"].apply(
    lambda text: calculate_token_length(text, secbert_tokenizer)
)

print("Token length columns added.")
print("combined_eda_df shape:", combined_eda_df.shape)

print("\nToken length columns:")
print([
    "distilbert_token_length",
    "roberta_token_length",
    "secbert_token_length"
])

print("\nPreview:")
display(
    combined_eda_df[
        [
            "text",
            "label",
            "split",
            "distilbert_token_length",
            "roberta_token_length",
            "secbert_token_length"
        ]
    ].head()
)

In [ ]:
# 3-Create token length summary by split and label

token_length_columns = [
    "distilbert_token_length",
    "roberta_token_length",
    "secbert_token_length"
]

token_length_summary_rows = []

for token_column in token_length_columns:
    model_name = token_column.replace("_token_length", "")

    summary_df = (
        combined_eda_df
        .groupby(["split", "label"])
        .agg(
            row_count=(token_column, "count"),
            min_token_length=(token_column, "min"),
            median_token_length=(token_column, "median"),
            mean_token_length=(token_column, "mean"),
            max_token_length=(token_column, "max")
        )
        .reset_index()
    )

    summary_df["model"] = model_name
    summary_df["label_name"] = summary_df["label"].map(label_mapping)

    token_length_summary_rows.append(summary_df)

token_length_summary_by_split_label = pd.concat(
    token_length_summary_rows,
    axis=0,
    ignore_index=True
)

token_length_summary_by_split_label = token_length_summary_by_split_label[
    [
        "model",
        "split",
        "label",
        "label_name",
        "row_count",
        "min_token_length",
        "median_token_length",
        "mean_token_length",
        "max_token_length"
    ]
].sort_values(["model", "split", "label"]).reset_index(drop=True)

token_length_summary_by_split_label["mean_token_length"] = (
    token_length_summary_by_split_label["mean_token_length"].round(2)
)

print("Token length summary by split and label:")
display(token_length_summary_by_split_label)

In [ ]:
# 4-Count prompts above token length thresholds

token_thresholds = [64, 128, 256, 512]

overall_token_threshold_rows = []

for token_column in token_length_columns:
    model_name = token_column.replace("_token_length", "")

    for threshold in token_thresholds:
        prompt_count_above_threshold = (combined_eda_df[token_column] > threshold).sum()
        percentage_above_threshold = round(
            (prompt_count_above_threshold / len(combined_eda_df)) * 100,
            2
        )

        overall_token_threshold_rows.append(
            {
                "model": model_name,
                "threshold": threshold,
                "prompt_count_above_threshold": prompt_count_above_threshold,
                "percentage_above_threshold": percentage_above_threshold
            }
        )

overall_token_threshold_summary = pd.DataFrame(overall_token_threshold_rows)

print("Overall token threshold summary:")
display(overall_token_threshold_summary)

In [ ]:
# 5-Count token thresholds by split and label

token_threshold_by_split_label_rows = []

for token_column in token_length_columns:
    model_name = token_column.replace("_token_length", "")

    for split_name in ["train", "test"]:
        for label_value in [0, 1]:
            subset_df = combined_eda_df[
                (combined_eda_df["split"] == split_name) &
                (combined_eda_df["label"] == label_value)
            ]

            for threshold in token_thresholds:
                prompt_count_above_threshold = (subset_df[token_column] > threshold).sum()
                percentage_above_threshold = round(
                    (prompt_count_above_threshold / len(subset_df)) * 100,
                    2
                )

                token_threshold_by_split_label_rows.append(
                    {
                        "model": model_name,
                        "split": split_name,
                        "label": label_value,
                        "label_name": label_mapping[label_value],
                        "threshold": threshold,
                        "prompt_count_above_threshold": prompt_count_above_threshold,
                        "percentage_above_threshold": percentage_above_threshold
                    }
                )

token_threshold_summary_by_split_label = pd.DataFrame(
    token_threshold_by_split_label_rows
)

print("Token threshold summary by split and label:")
display(token_threshold_summary_by_split_label)

In [ ]:
# 6-Save token length analysis outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

text_length_with_token_lengths_path = TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
token_length_summary_path = TABLES_DIR / "token_length_summary_by_split_label.csv"
overall_token_threshold_summary_path = TABLES_DIR / "overall_token_threshold_summary.csv"
token_threshold_summary_by_split_label_path = TABLES_DIR / "token_threshold_summary_by_split_label.csv"

combined_eda_df.to_csv(text_length_with_token_lengths_path, index=False)
token_length_summary_by_split_label.to_csv(token_length_summary_path, index=False)
overall_token_threshold_summary.to_csv(overall_token_threshold_summary_path, index=False)
token_threshold_summary_by_split_label.to_csv(token_threshold_summary_by_split_label_path, index=False)

print("Text length with token lengths saved to:", text_length_with_token_lengths_path)
print("File exists:", text_length_with_token_lengths_path.exists())

print("\nToken length summary saved to:", token_length_summary_path)
print("File exists:", token_length_summary_path.exists())

print("\nOverall token threshold summary saved to:", overall_token_threshold_summary_path)
print("File exists:", overall_token_threshold_summary_path.exists())

print("\nToken threshold summary by split and label saved to:", token_threshold_summary_by_split_label_path)
print("File exists:", token_threshold_summary_by_split_label_path.exists())

#Part-09: Unusual Character Detection

In [ ]:
# 1-Create unusual-character flags

import unicodedata

ZERO_WIDTH_CHARACTERS = [
    "\u200b",  # zero-width space
    "\u200c",  # zero-width non-joiner
    "\u200d",  # zero-width joiner
    "\ufeff",  # zero-width no-break space / BOM
    "\u2060"   # word joiner
]

def has_other_control_character(text):
    text = str(text)

    allowed_control_characters = {"\n", "\t", "\r"}

    for character in text:
        if unicodedata.category(character) == "Cc" and character not in allowed_control_characters:
            return True

    return False

def has_zero_width_character(text):
    text = str(text)
    return any(character in text for character in ZERO_WIDTH_CHARACTERS)

unusual_character_analysis_df = combined_eda_df.copy()

unusual_character_analysis_df["has_newline"] = unusual_character_analysis_df["text"].astype(str).str.contains("\n", regex=False)
unusual_character_analysis_df["has_tab"] = unusual_character_analysis_df["text"].astype(str).str.contains("\t", regex=False)
unusual_character_analysis_df["has_carriage_return"] = unusual_character_analysis_df["text"].astype(str).str.contains("\r", regex=False)
unusual_character_analysis_df["has_repeated_spaces"] = unusual_character_analysis_df["text"].astype(str).str.contains(r" {2,}", regex=True)
unusual_character_analysis_df["has_zero_width_or_invisible"] = unusual_character_analysis_df["text"].apply(has_zero_width_character)
unusual_character_analysis_df["has_other_control_character"] = unusual_character_analysis_df["text"].apply(has_other_control_character)

unusual_flag_columns = [
    "has_newline",
    "has_tab",
    "has_carriage_return",
    "has_repeated_spaces",
    "has_zero_width_or_invisible",
    "has_other_control_character"
]

unusual_character_analysis_df["has_any_unusual_character"] = unusual_character_analysis_df[
    unusual_flag_columns
].any(axis=1)

print("Unusual-character flags created.")
print("unusual_character_analysis_df shape:", unusual_character_analysis_df.shape)

print("\nFlag columns:")
print(unusual_flag_columns + ["has_any_unusual_character"])

In [ ]:
# 2-Summarise unusual-character types

unusual_character_summary = pd.DataFrame([
    {
        "character_issue": "newline",
        "prompt_count": unusual_character_analysis_df["has_newline"].sum()
    },
    {
        "character_issue": "tab",
        "prompt_count": unusual_character_analysis_df["has_tab"].sum()
    },
    {
        "character_issue": "carriage_return",
        "prompt_count": unusual_character_analysis_df["has_carriage_return"].sum()
    },
    {
        "character_issue": "repeated_spaces",
        "prompt_count": unusual_character_analysis_df["has_repeated_spaces"].sum()
    },
    {
        "character_issue": "zero_width_or_invisible",
        "prompt_count": unusual_character_analysis_df["has_zero_width_or_invisible"].sum()
    },
    {
        "character_issue": "other_control_character",
        "prompt_count": unusual_character_analysis_df["has_other_control_character"].sum()
    },
    {
        "character_issue": "any_unusual_character",
        "prompt_count": unusual_character_analysis_df["has_any_unusual_character"].sum()
    }
])

unusual_character_summary["percentage_of_dataset"] = (
    unusual_character_summary["prompt_count"] / len(unusual_character_analysis_df) * 100
).round(2)

print("Unusual-character summary:")
display(unusual_character_summary)

In [ ]:
# 3-Summarise unusual characters by split and label

unusual_character_count_by_split = (
    unusual_character_analysis_df
    .groupby("split")["has_any_unusual_character"]
    .sum()
    .reset_index(name="unusual_character_prompt_count")
)

unusual_character_count_by_split["total_prompts"] = (
    unusual_character_analysis_df
    .groupby("split")["text"]
    .count()
    .values
)

unusual_character_count_by_split["percentage"] = (
    unusual_character_count_by_split["unusual_character_prompt_count"] /
    unusual_character_count_by_split["total_prompts"] * 100
).round(2)

unusual_character_count_by_label = (
    unusual_character_analysis_df
    .groupby("label")["has_any_unusual_character"]
    .sum()
    .reset_index(name="unusual_character_prompt_count")
)

unusual_character_count_by_label["label_name"] = unusual_character_count_by_label["label"].map(label_mapping)

unusual_character_count_by_label["total_prompts"] = (
    unusual_character_analysis_df
    .groupby("label")["text"]
    .count()
    .values
)

unusual_character_count_by_label["percentage"] = (
    unusual_character_count_by_label["unusual_character_prompt_count"] /
    unusual_character_count_by_label["total_prompts"] * 100
).round(2)

unusual_character_count_by_label = unusual_character_count_by_label[
    ["label", "label_name", "unusual_character_prompt_count", "total_prompts", "percentage"]
]

print("Unusual-character count by split:")
display(unusual_character_count_by_split)

print("\nUnusual-character count by label:")
display(unusual_character_count_by_label)

In [ ]:
# 4-Extract unusual-character examples

unusual_character_examples = unusual_character_analysis_df[
    unusual_character_analysis_df["has_any_unusual_character"]
].copy()

unusual_character_examples["label_name"] = unusual_character_examples["label"].map(label_mapping)

unusual_character_examples = unusual_character_examples[
    [
        "split",
        "label",
        "label_name",
        "text",
        "has_newline",
        "has_tab",
        "has_carriage_return",
        "has_repeated_spaces",
        "has_zero_width_or_invisible",
        "has_other_control_character",
        "has_any_unusual_character"
    ]
].reset_index(drop=True)

print("Number of unusual-character examples:", len(unusual_character_examples))
display(unusual_character_examples.head(10))

In [ ]:
# 5-Save unusual-character analysis outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

unusual_character_summary_path = TABLES_DIR / "unusual_character_summary.csv"
unusual_character_count_by_split_path = TABLES_DIR / "unusual_character_count_by_split.csv"
unusual_character_count_by_label_path = TABLES_DIR / "unusual_character_count_by_label.csv"
unusual_character_examples_path = TABLES_DIR / "unusual_character_examples.csv"

unusual_character_summary.to_csv(unusual_character_summary_path, index=False)
unusual_character_count_by_split.to_csv(unusual_character_count_by_split_path, index=False)
unusual_character_count_by_label.to_csv(unusual_character_count_by_label_path, index=False)
unusual_character_examples.to_csv(unusual_character_examples_path, index=False)

print("Unusual-character summary saved to:", unusual_character_summary_path)
print("File exists:", unusual_character_summary_path.exists())

print("\nUnusual-character count by split saved to:", unusual_character_count_by_split_path)
print("File exists:", unusual_character_count_by_split_path.exists())

print("\nUnusual-character count by label saved to:", unusual_character_count_by_label_path)
print("File exists:", unusual_character_count_by_label_path.exists())

print("\nUnusual-character examples saved to:", unusual_character_examples_path)
print("File exists:", unusual_character_examples_path.exists())

#Part-10: Suspicious Pattern Analysis

In [ ]:
# 1-Define suspicious pattern terms

import re

prompt_injection_terms = [
    "ignore",
    "forget",
    "previous",
    "instruction",
    "instructions",
    "system",
    "developer",
    "prompt",
    "jailbreak",
    "dan",
    "chatgpt",
    "mode",
    "override",
    "bypass"
]

code_like_patterns = {
    "python_code": r"\bdef\b|\bimport\b|\bprint\s*\(",
    "sql_code": r"\bSELECT\b|\bINSERT\b|\bUPDATE\b|\bDELETE\b|\bDROP\b|\bFROM\b|\bWHERE\b",
    "html_or_xml": r"<[^>]+>",
    "code_block": r"```",
    "programming_language": r"\bpython\b|\bjava\b|\bc\+\+\b|\bjavascript\b|\bsql\b|\blinux\b|\bterminal\b"
}

command_like_patterns = {
    "ignore_or_forget_instruction": r"\b(ignore|forget)\b.*\b(instruction|instructions|previous|above|all)\b",
    "act_as_role_instruction": r"\b(act as|pretend to be|you are now|roleplay as)\b",
    "new_task_or_override": r"\b(new task|override|bypass|disregard)\b",
    "execution_command": r"\b(write|generate|create|print|execute|run|translate|summarise|summarize|answer|explain)\b",
    "dan_or_jailbreak": r"\b(dan|jailbreak)\b"
}

print("Suspicious pattern term lists created.")
print("Prompt-injection terms:", len(prompt_injection_terms))
print("Code-like pattern groups:", len(code_like_patterns))
print("Command-like pattern groups:", len(command_like_patterns))

In [ ]:
# 2-Count prompt-injection terms by label

suspicious_pattern_analysis_df = combined_eda_df.copy()
suspicious_pattern_analysis_df["label_name"] = suspicious_pattern_analysis_df["label"].map(label_mapping)

prompt_injection_term_rows = []

for label_value in [0, 1]:
    label_subset = suspicious_pattern_analysis_df[
        suspicious_pattern_analysis_df["label"] == label_value
    ]

    for term in prompt_injection_terms:
        term_pattern = rf"\b{re.escape(term)}\b"

        prompt_count = label_subset["text"].astype(str).str.contains(
            term_pattern,
            case=False,
            regex=True
        ).sum()

        occurrence_count = label_subset["text"].astype(str).str.count(
            term_pattern,
            flags=re.IGNORECASE
        ).sum()

        prompt_percentage = round((prompt_count / len(label_subset)) * 100, 2)

        prompt_injection_term_rows.append(
            {
                "label": label_value,
                "label_name": label_mapping[label_value],
                "term": term,
                "prompt_count": prompt_count,
                "occurrence_count": occurrence_count,
                "prompt_percentage": prompt_percentage
            }
        )

prompt_injection_term_counts_by_label = pd.DataFrame(prompt_injection_term_rows)

print("Prompt-injection term counts by label:")
display(prompt_injection_term_counts_by_label)

In [ ]:
# 3-Check URL patterns

url_pattern = r"https?://|www\.|[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"

suspicious_pattern_analysis_df["has_url"] = suspicious_pattern_analysis_df["text"].astype(str).str.contains(
    url_pattern,
    case=False,
    regex=True
)

url_count_summary_by_label = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["has_url"]
    .sum()
    .reset_index(name="url_prompt_count")
)

url_count_summary_by_label["total_prompts"] = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["text"]
    .count()
    .values
)

url_count_summary_by_label["percentage"] = (
    url_count_summary_by_label["url_prompt_count"] /
    url_count_summary_by_label["total_prompts"] * 100
).round(2)

url_examples = suspicious_pattern_analysis_df[
    suspicious_pattern_analysis_df["has_url"]
][["split", "label", "label_name", "text", "has_url"]].reset_index(drop=True)

print("URL count summary by label:")
display(url_count_summary_by_label)

print("\nNumber of URL examples:", len(url_examples))
display(url_examples.head())

In [ ]:
# 4-Detect code-like patterns with balanced matching

code_like_patterns = {
    "python_code": r"\bdef\s+\w+\s*\(|\bimport\s+\w+|\bfrom\s+\w+\s+import\b|\bprint\s*\(|\bpython\s+interpreter\b|\bpython\b",
    "sql_code": r"\bsql\b|\bdatabase\b|\bselect\b.+\bfrom\b|\binsert\s+into\b|\bupdate\s+\w+\s+set\b|\bdelete\s+from\b|\bdrop\s+(table|database)\b",
    "html_or_xml": r"<[^>]+>",
    "code_block": r"```",
    "programming_language": r"\bc\+\+\b|\bjava\b|\bjavascript\b|\blinux\b|\bterminal\b|\bcode\b"
}

for pattern_name, pattern_regex in code_like_patterns.items():
    suspicious_pattern_analysis_df[f"has_code_like_{pattern_name}"] = (
        suspicious_pattern_analysis_df["text"]
        .astype(str)
        .str.contains(pattern_regex, case=False, regex=True)
    )

code_like_flag_columns = [
    f"has_code_like_{pattern_name}" for pattern_name in code_like_patterns.keys()
]

suspicious_pattern_analysis_df["has_any_code_like_pattern"] = suspicious_pattern_analysis_df[
    code_like_flag_columns
].any(axis=1)

code_like_overall_summary_by_label = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["has_any_code_like_pattern"]
    .sum()
    .reset_index(name="code_like_prompt_count")
)

code_like_overall_summary_by_label["total_prompts"] = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["text"]
    .count()
    .values
)

code_like_overall_summary_by_label["percentage"] = (
    code_like_overall_summary_by_label["code_like_prompt_count"] /
    code_like_overall_summary_by_label["total_prompts"] * 100
).round(2)

code_like_examples = suspicious_pattern_analysis_df[
    suspicious_pattern_analysis_df["has_any_code_like_pattern"]
][
    ["split", "label", "label_name", "text"] +
    code_like_flag_columns +
    ["has_any_code_like_pattern"]
].reset_index(drop=True)

print("Final code-like overall summary by label:")
display(code_like_overall_summary_by_label)

print("\nNumber of final code-like examples:", len(code_like_examples))
display(code_like_examples.head(10))

In [ ]:
# 5-Detect command-like patterns

for pattern_name, pattern_regex in command_like_patterns.items():
    suspicious_pattern_analysis_df[f"has_command_like_{pattern_name}"] = (
        suspicious_pattern_analysis_df["text"]
        .astype(str)
        .str.contains(pattern_regex, case=False, regex=True)
    )

command_like_flag_columns = [
    f"has_command_like_{pattern_name}" for pattern_name in command_like_patterns.keys()
]

suspicious_pattern_analysis_df["has_any_command_like_pattern"] = suspicious_pattern_analysis_df[
    command_like_flag_columns
].any(axis=1)

command_like_overall_summary_by_label = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["has_any_command_like_pattern"]
    .sum()
    .reset_index(name="command_like_prompt_count")
)

command_like_overall_summary_by_label["total_prompts"] = (
    suspicious_pattern_analysis_df
    .groupby(["label", "label_name"])["text"]
    .count()
    .values
)

command_like_overall_summary_by_label["percentage"] = (
    command_like_overall_summary_by_label["command_like_prompt_count"] /
    command_like_overall_summary_by_label["total_prompts"] * 100
).round(2)

command_like_examples = suspicious_pattern_analysis_df[
    suspicious_pattern_analysis_df["has_any_command_like_pattern"]
][
    ["split", "label", "label_name", "text"] +
    command_like_flag_columns +
    ["has_any_command_like_pattern"]
].reset_index(drop=True)

print("Command-like overall summary by label:")
display(command_like_overall_summary_by_label)

print("\nNumber of command-like examples:", len(command_like_examples))
display(command_like_examples.head(10))

In [ ]:
# 6-Create pattern frequency comparison by label

pattern_frequency_rows = []

summary_items = [
    {
        "pattern_name": "has_url",
        "column_name": "has_url"
    },
    {
        "pattern_name": "has_any_code_like_pattern",
        "column_name": "has_any_code_like_pattern"
    },
    {
        "pattern_name": "has_any_command_like_pattern",
        "column_name": "has_any_command_like_pattern"
    }
]

for term in prompt_injection_terms:
    column_name = f"contains_term_{term}"

    suspicious_pattern_analysis_df[column_name] = suspicious_pattern_analysis_df["text"].astype(str).str.contains(
        rf"\b{re.escape(term)}\b",
        case=False,
        regex=True
    )

    summary_items.append(
        {
            "pattern_name": term,
            "column_name": column_name
        }
    )

for item in summary_items:
    pattern_name = item["pattern_name"]
    column_name = item["column_name"]

    safe_subset = suspicious_pattern_analysis_df[
        suspicious_pattern_analysis_df["label"] == 0
    ]

    injection_subset = suspicious_pattern_analysis_df[
        suspicious_pattern_analysis_df["label"] == 1
    ]

    safe_count = safe_subset[column_name].sum()
    injection_count = injection_subset[column_name].sum()

    safe_percentage = round((safe_count / len(safe_subset)) * 100, 2)
    injection_percentage = round((injection_count / len(injection_subset)) * 100, 2)

    pattern_frequency_rows.append(
        {
            "pattern_name": pattern_name,
            "safe_prompt_count": safe_count,
            "safe_percentage": safe_percentage,
            "injection_prompt_count": injection_count,
            "injection_percentage": injection_percentage,
            "percentage_difference_injection_minus_safe": round(
                injection_percentage - safe_percentage,
                2
            )
        }
    )

pattern_frequency_comparison_by_label = pd.DataFrame(pattern_frequency_rows)

pattern_frequency_comparison_by_label = pattern_frequency_comparison_by_label.sort_values(
    "percentage_difference_injection_minus_safe",
    ascending=False
).reset_index(drop=True)

print("Pattern frequency comparison by label:")
display(pattern_frequency_comparison_by_label)

In [ ]:
# 7-Save suspicious-pattern analysis outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

suspicious_pattern_analysis_path = TABLES_DIR / "suspicious_pattern_analysis_with_flags.csv"
prompt_injection_term_counts_path = TABLES_DIR / "prompt_injection_term_counts_by_label.csv"
url_count_summary_path = TABLES_DIR / "url_count_summary_by_label.csv"
url_examples_path = TABLES_DIR / "url_examples.csv"
code_like_summary_path = TABLES_DIR / "code_like_overall_summary_by_label.csv"
code_like_examples_path = TABLES_DIR / "code_like_examples.csv"
command_like_summary_path = TABLES_DIR / "command_like_overall_summary_by_label.csv"
command_like_examples_path = TABLES_DIR / "command_like_examples.csv"
pattern_frequency_comparison_path = TABLES_DIR / "pattern_frequency_comparison_by_label.csv"

suspicious_pattern_analysis_df.to_csv(suspicious_pattern_analysis_path, index=False)
prompt_injection_term_counts_by_label.to_csv(prompt_injection_term_counts_path, index=False)
url_count_summary_by_label.to_csv(url_count_summary_path, index=False)
url_examples.to_csv(url_examples_path, index=False)
code_like_overall_summary_by_label.to_csv(code_like_summary_path, index=False)
code_like_examples.to_csv(code_like_examples_path, index=False)
command_like_overall_summary_by_label.to_csv(command_like_summary_path, index=False)
command_like_examples.to_csv(command_like_examples_path, index=False)
pattern_frequency_comparison_by_label.to_csv(pattern_frequency_comparison_path, index=False)

print("Suspicious-pattern outputs saved:")

print("suspicious_pattern_analysis_with_flags.csv:", suspicious_pattern_analysis_path.exists())
print("prompt_injection_term_counts_by_label.csv:", prompt_injection_term_counts_path.exists())
print("url_count_summary_by_label.csv:", url_count_summary_path.exists())
print("url_examples.csv:", url_examples_path.exists())
print("code_like_overall_summary_by_label.csv:", code_like_summary_path.exists())
print("code_like_examples.csv:", code_like_examples_path.exists())
print("command_like_overall_summary_by_label.csv:", command_like_summary_path.exists())
print("command_like_examples.csv:", command_like_examples_path.exists())
print("pattern_frequency_comparison_by_label.csv:", pattern_frequency_comparison_path.exists())

#Part-11: Outlier Analysis

In [ ]:
# 1-Identify shortest prompts by word count

shortest_prompt_examples = (
    combined_eda_df
    .copy()
    .sort_values(["word_count", "character_length"], ascending=[True, True])
    .head(15)
)

shortest_prompt_examples["label_name"] = shortest_prompt_examples["label"].map(label_mapping)

shortest_prompt_examples = shortest_prompt_examples[
    [
        "split",
        "label",
        "label_name",
        "text",
        "character_length",
        "word_count"
    ]
].reset_index(drop=True)

shortest_prompt_label_summary = (
    shortest_prompt_examples
    .groupby(["label", "label_name"])
    .size()
    .reset_index(name="shortest_prompt_count")
)

print("Shortest prompt examples:")
display(shortest_prompt_examples)

print("\nShortest prompt label summary:")
display(shortest_prompt_label_summary)

In [ ]:
# 2-Identify longest prompts by character length

longest_prompt_examples = (
    combined_eda_df
    .copy()
    .sort_values(["character_length", "word_count"], ascending=[False, False])
    .head(15)
)

longest_prompt_examples["label_name"] = longest_prompt_examples["label"].map(label_mapping)

longest_prompt_examples = longest_prompt_examples[
    [
        "split",
        "label",
        "label_name",
        "text",
        "character_length",
        "word_count"
    ]
].reset_index(drop=True)

longest_prompt_label_summary = (
    longest_prompt_examples
    .groupby(["label", "label_name"])
    .size()
    .reset_index(name="longest_prompt_count")
)

print("Longest prompt examples:")
display(longest_prompt_examples)

print("\nLongest prompt label summary:")
display(longest_prompt_label_summary)

In [ ]:
# 3-Identify prompts exceeding 512 tokens for any tokenizer

token_length_columns = [
    "distilbert_token_length",
    "roberta_token_length",
    "secbert_token_length"
]

combined_eda_df["max_token_length_across_tokenizers"] = combined_eda_df[token_length_columns].max(axis=1)

extreme_token_length_examples = combined_eda_df[
    (combined_eda_df["distilbert_token_length"] > 512) |
    (combined_eda_df["roberta_token_length"] > 512) |
    (combined_eda_df["secbert_token_length"] > 512)
].copy()

extreme_token_length_examples["label_name"] = extreme_token_length_examples["label"].map(label_mapping)

extreme_token_length_examples = extreme_token_length_examples[
    [
        "split",
        "label",
        "label_name",
        "text",
        "character_length",
        "word_count",
        "distilbert_token_length",
        "roberta_token_length",
        "secbert_token_length",
        "max_token_length_across_tokenizers"
    ]
].sort_values("max_token_length_across_tokenizers", ascending=False).reset_index(drop=True)

extreme_token_label_summary = (
    extreme_token_length_examples
    .groupby(["label", "label_name"])
    .size()
    .reset_index(name="extreme_token_prompt_count")
)

print("Extreme token-length examples:")
display(extreme_token_length_examples)

print("\nExtreme token-length label summary:")
display(extreme_token_label_summary)

In [ ]:
# 4-Create outlier summary

outlier_summary = pd.DataFrame([
    {
        "outlier_check": "reviewed_shortest_prompts",
        "value": len(shortest_prompt_examples),
        "interpretation": "Manual review set of shortest prompts"
    },
    {
        "outlier_check": "reviewed_longest_prompts",
        "value": len(longest_prompt_examples),
        "interpretation": "Manual review set of longest prompts"
    },
    {
        "outlier_check": "prompts_above_512_any_tokenizer",
        "value": len(extreme_token_length_examples),
        "interpretation": "Prompts requiring truncation during transformer fine-tuning"
    },
    {
        "outlier_check": "max_character_length",
        "value": combined_eda_df["character_length"].max(),
        "interpretation": "Longest prompt by character length"
    },
    {
        "outlier_check": "max_word_count",
        "value": combined_eda_df["word_count"].max(),
        "interpretation": "Longest prompt by word count"
    },
    {
        "outlier_check": "max_distilbert_token_length",
        "value": combined_eda_df["distilbert_token_length"].max(),
        "interpretation": "Maximum DistilBERT token length"
    },
    {
        "outlier_check": "max_roberta_token_length",
        "value": combined_eda_df["roberta_token_length"].max(),
        "interpretation": "Maximum RoBERTa token length"
    },
    {
        "outlier_check": "max_secbert_token_length",
        "value": combined_eda_df["secbert_token_length"].max(),
        "interpretation": "Maximum SecBERT token length"
    }
])

print("Outlier summary:")
display(outlier_summary)

In [ ]:
# 5-Save outlier analysis outputs

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

shortest_prompt_examples_path = TABLES_DIR / "shortest_prompt_examples.csv"
longest_prompt_examples_path = TABLES_DIR / "longest_prompt_examples.csv"
extreme_token_length_examples_path = TABLES_DIR / "extreme_token_length_examples.csv"
outlier_summary_path = TABLES_DIR / "outlier_summary.csv"

shortest_prompt_examples.to_csv(shortest_prompt_examples_path, index=False)
longest_prompt_examples.to_csv(longest_prompt_examples_path, index=False)
extreme_token_length_examples.to_csv(extreme_token_length_examples_path, index=False)
outlier_summary.to_csv(outlier_summary_path, index=False)

print("Outlier analysis outputs saved:")

print("shortest_prompt_examples.csv:", shortest_prompt_examples_path.exists())
print("longest_prompt_examples.csv:", longest_prompt_examples_path.exists())
print("extreme_token_length_examples.csv:", extreme_token_length_examples_path.exists())
print("outlier_summary.csv:", outlier_summary_path.exists())

#Part-12: Data Risk Analysis

In [ ]:
# 1-Extract key risk metrics from previous analysis outputs

missing_invalid_issue_count = int(data_quality_validation_summary["issue_count"].sum())
duplicate_leakage_issue_count = int(duplicate_leakage_summary["issue_count"].sum())

train_safe_percentage = class_balance_comparison[
    (class_balance_comparison["split"] == "train") &
    (class_balance_comparison["label"] == 0)
]["percentage"].iloc[0]

train_injection_percentage = class_balance_comparison[
    (class_balance_comparison["split"] == "train") &
    (class_balance_comparison["label"] == 1)
]["percentage"].iloc[0]

prompts_above_512_any_tokenizer = int(
    outlier_summary[
        outlier_summary["outlier_check"] == "prompts_above_512_any_tokenizer"
    ]["value"].iloc[0]
)

max_character_length = int(
    outlier_summary[
        outlier_summary["outlier_check"] == "max_character_length"
    ]["value"].iloc[0]
)

max_word_count = int(
    outlier_summary[
        outlier_summary["outlier_check"] == "max_word_count"
    ]["value"].iloc[0]
)

any_unusual_character_count = int(
    unusual_character_summary[
        unusual_character_summary["character_issue"] == "any_unusual_character"
    ]["prompt_count"].iloc[0]
)

any_unusual_character_percentage = float(
    unusual_character_summary[
        unusual_character_summary["character_issue"] == "any_unusual_character"
    ]["percentage_of_dataset"].iloc[0]
)

safe_command_like_count = int(
    command_like_overall_summary_by_label[
        command_like_overall_summary_by_label["label"] == 0
    ]["command_like_prompt_count"].iloc[0]
)

injection_command_like_count = int(
    command_like_overall_summary_by_label[
        command_like_overall_summary_by_label["label"] == 1
    ]["command_like_prompt_count"].iloc[0]
)

safe_code_like_count = int(
    code_like_overall_summary_by_label[
        code_like_overall_summary_by_label["label"] == 0
    ]["code_like_prompt_count"].iloc[0]
)

injection_code_like_count = int(
    code_like_overall_summary_by_label[
        code_like_overall_summary_by_label["label"] == 1
    ]["code_like_prompt_count"].iloc[0]
)

safe_url_count = int(
    url_count_summary_by_label[
        url_count_summary_by_label["label"] == 0
    ]["url_prompt_count"].iloc[0]
)

injection_url_count = int(
    url_count_summary_by_label[
        url_count_summary_by_label["label"] == 1
    ]["url_prompt_count"].iloc[0]
)

print("Key risk metrics extracted.")
print("Missing/invalid issue count:", missing_invalid_issue_count)
print("Duplicate/leakage issue count:", duplicate_leakage_issue_count)
print("Train SAFE percentage:", train_safe_percentage)
print("Train INJECTION percentage:", train_injection_percentage)
print("Prompts above 512 tokens for any tokenizer:", prompts_above_512_any_tokenizer)
print("Max character length:", max_character_length)
print("Max word count:", max_word_count)
print("Any unusual-character prompts:", any_unusual_character_count)
print("Any unusual-character percentage:", any_unusual_character_percentage)
print("Command-like prompts - SAFE:", safe_command_like_count)
print("Command-like prompts - INJECTION:", injection_command_like_count)
print("Code-like prompts - SAFE:", safe_code_like_count)
print("Code-like prompts - INJECTION:", injection_code_like_count)
print("URL prompts - SAFE:", safe_url_count)
print("URL prompts - INJECTION:", injection_url_count)

In [ ]:
# 2-Create final data risk summary

data_risk_summary = pd.DataFrame([
    {
        "risk_area": "missing_or_invalid_text",
        "risk_level": "Low",
        "evidence": f"{missing_invalid_issue_count} missing, empty, whitespace-only, non-string, or symbol-only text issues found.",
        "part_2_decision": "No row removal required for missing or invalid text."
    },
    {
        "risk_area": "duplicates_and_train_test_leakage",
        "risk_level": "Low",
        "evidence": f"{duplicate_leakage_issue_count} duplicate, conflicting-label, or train-test leakage issues found.",
        "part_2_decision": "Keep original train/test split. No duplicate-based removal required."
    },
    {
        "risk_area": "class_imbalance",
        "risk_level": "Medium",
        "evidence": f"Training split is imbalanced: SAFE = {train_safe_percentage}%, INJECTION = {train_injection_percentage}%.",
        "part_2_decision": "Use stratified validation split and report Precision, Recall, F1-score, and AUC-ROC."
    },
    {
        "risk_area": "long_text_and_token_truncation",
        "risk_level": "Medium",
        "evidence": f"{prompts_above_512_any_tokenizer} prompts exceed 512 tokens for at least one tokenizer. Max character length = {max_character_length}; max word count = {max_word_count}.",
        "part_2_decision": "Use max_length=512 and truncation=True during tokenizer preparation."
    },
    {
        "risk_area": "unusual_characters",
        "risk_level": "Medium",
        "evidence": f"{any_unusual_character_count} prompts ({any_unusual_character_percentage}%) contain unusual characters such as newline, repeated spaces, or zero-width characters.",
        "part_2_decision": "Do not automatically remove these prompts. Keep raw text but document the risk."
    },
    {
        "risk_area": "url_patterns",
        "risk_level": "Low",
        "evidence": f"URL prompts found: SAFE = {safe_url_count}, INJECTION = {injection_url_count}.",
        "part_2_decision": "No URL-specific preprocessing required."
    },
    {
        "risk_area": "code_like_patterns",
        "risk_level": "Low",
        "evidence": f"Code-like prompts found: SAFE = {safe_code_like_count}, INJECTION = {injection_code_like_count}.",
        "part_2_decision": "Keep these prompts because they may represent valid prompt-injection behaviour."
    },
    {
        "risk_area": "command_like_patterns",
        "risk_level": "Medium",
        "evidence": f"Command-like prompts found: SAFE = {safe_command_like_count}, INJECTION = {injection_command_like_count}.",
        "part_2_decision": "Do not use rules for classification. Keep patterns only as exploratory evidence."
    },
    {
        "risk_area": "outliers",
        "risk_level": "Medium",
        "evidence": "Shortest prompts include mostly SAFE prompts, while the 15 longest prompts are all INJECTION.",
        "part_2_decision": "Do not remove outliers automatically. Handle long prompts with tokenizer truncation."
    }
])

print("Data risk summary:")
display(data_risk_summary)

In [ ]:
# 3-Create final decisions for Part 2 data preparation

part_2_preparation_decisions = pd.DataFrame([
    {
        "decision_area": "raw_data_source",
        "decision": "Use the original raw train and test parquet files from the GitHub repository.",
        "reason": "The dataset loaded correctly and passed basic validation."
    },
    {
        "decision_area": "train_test_split",
        "decision": "Keep the official train and test split unchanged.",
        "reason": "No train-test leakage was detected."
    },
    {
        "decision_area": "validation_split",
        "decision": "Create validation data only from the training split.",
        "reason": "The official test set must remain untouched for final evaluation."
    },
    {
        "decision_area": "stratification",
        "decision": "Use stratified splitting for validation.",
        "reason": "Training data has moderate class imbalance."
    },
    {
        "decision_area": "text_cleaning",
        "decision": "Do not aggressively clean, rewrite, or remove prompt text.",
        "reason": "Prompt-injection signals may depend on exact wording, formatting, and unusual characters."
    },
    {
        "decision_area": "tokenization",
        "decision": "Use max_length=512, padding='max_length', and truncation=True.",
        "reason": "A small number of prompts exceed the transformer sequence limit."
    },
    {
        "decision_area": "label_handling",
        "decision": "Keep binary labels unchanged: 0 = SAFE, 1 = INJECTION.",
        "reason": "Labels are valid and no conflicting labels were found."
    },
    {
        "decision_area": "processed_outputs",
        "decision": "Save processed datasets inside data/processed in the GitHub repository.",
        "reason": "This matches the project workflow and supports later GitHub commit/push."
    }
])

print("Part 2 preparation decisions:")
display(part_2_preparation_decisions)

In [ ]:
# 4-Save risk summary and Part 2 preparation decisions

TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

data_risk_summary_path = TABLES_DIR / "data_risk_summary.csv"
part_2_preparation_decisions_path = TABLES_DIR / "part_2_preparation_decisions.csv"

data_risk_summary.to_csv(data_risk_summary_path, index=False)
part_2_preparation_decisions.to_csv(part_2_preparation_decisions_path, index=False)

print("Data risk summary saved:")
print("data_risk_summary.csv:", data_risk_summary_path.exists())

print("\nPart 2 preparation decisions saved:")
print("part_2_preparation_decisions.csv:", part_2_preparation_decisions_path.exists())

#Part-13- Necessary Visualization from Data Analysis

In [ ]:
# 1-Create class distribution figure

import matplotlib.pyplot as plt

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

class_distribution_plot_df = (
    class_balance_comparison[
        class_balance_comparison["split"].isin(["train", "test"])
    ]
    .pivot(index="split", columns="label_name", values="count")
    .loc[["train", "test"]]
)

ax = class_distribution_plot_df.plot(kind="bar", figsize=(8, 5))

plt.title("Class Distribution by Split")
plt.xlabel("Dataset Split")
plt.ylabel("Prompt Count")
plt.xticks(rotation=0)
plt.legend(title="Label")
plt.tight_layout()

class_distribution_figure_path = FIGURES_DIR / "class_distribution_by_split.png"
plt.savefig(class_distribution_figure_path, dpi=300)
plt.show()

print("Class distribution figure saved to:", class_distribution_figure_path)
print("File exists:", class_distribution_figure_path.exists())

In [ ]:
# 2-Create character-length distribution figure

safe_character_lengths = combined_eda_df[
    combined_eda_df["label"] == 0
]["character_length"]

injection_character_lengths = combined_eda_df[
    combined_eda_df["label"] == 1
]["character_length"]

plt.figure(figsize=(8, 5))

plt.hist(safe_character_lengths, bins=40, alpha=0.7, label="SAFE")
plt.hist(injection_character_lengths, bins=40, alpha=0.7, label="INJECTION")

plt.title("Character-Length Distribution by Class")
plt.xlabel("Character Length")
plt.ylabel("Prompt Count")
plt.legend()
plt.tight_layout()

character_length_figure_path = FIGURES_DIR / "character_length_distribution_by_class.png"
plt.savefig(character_length_figure_path, dpi=300)
plt.show()

print("Character-length figure saved to:", character_length_figure_path)
print("File exists:", character_length_figure_path.exists())

In [ ]:
# 3-Create word-count distribution figure

safe_word_counts = combined_eda_df[
    combined_eda_df["label"] == 0
]["word_count"]

injection_word_counts = combined_eda_df[
    combined_eda_df["label"] == 1
]["word_count"]

plt.figure(figsize=(8, 5))

plt.hist(safe_word_counts, bins=40, alpha=0.7, label="SAFE")
plt.hist(injection_word_counts, bins=40, alpha=0.7, label="INJECTION")

plt.title("Word-Count Distribution by Class")
plt.xlabel("Word Count")
plt.ylabel("Prompt Count")
plt.legend()
plt.tight_layout()

word_count_figure_path = FIGURES_DIR / "word_count_distribution_by_class.png"
plt.savefig(word_count_figure_path, dpi=300)
plt.show()

print("Word-count figure saved to:", word_count_figure_path)
print("File exists:", word_count_figure_path.exists())

In [ ]:
# 4-Create token-length distribution figure

plt.figure(figsize=(8, 5))

plt.hist(combined_eda_df["distilbert_token_length"], bins=40, alpha=0.6, label="DistilBERT")
plt.hist(combined_eda_df["roberta_token_length"], bins=40, alpha=0.6, label="RoBERTa")
plt.hist(combined_eda_df["secbert_token_length"], bins=40, alpha=0.6, label="SecBERT")

plt.axvline(512, linestyle="--", label="512-token limit")

plt.title("Token-Length Distribution by Tokenizer")
plt.xlabel("Token Length")
plt.ylabel("Prompt Count")
plt.legend()
plt.tight_layout()

token_length_figure_path = FIGURES_DIR / "token_length_distribution_by_tokenizer.png"
plt.savefig(token_length_figure_path, dpi=300)
plt.show()

print("Token-length figure saved to:", token_length_figure_path)
print("File exists:", token_length_figure_path.exists())

In [ ]:
# 5-Create median length comparison figure

median_length_comparison = (
    combined_eda_df
    .groupby("label")
    .agg(
        median_character_length=("character_length", "median"),
        median_word_count=("word_count", "median"),
        median_distilbert_tokens=("distilbert_token_length", "median"),
        median_roberta_tokens=("roberta_token_length", "median"),
        median_secbert_tokens=("secbert_token_length", "median")
    )
    .reset_index()
)

median_length_comparison["label_name"] = median_length_comparison["label"].map(label_mapping)

median_length_plot_df = median_length_comparison.set_index("label_name")[
    [
        "median_character_length",
        "median_word_count",
        "median_distilbert_tokens",
        "median_roberta_tokens",
        "median_secbert_tokens"
    ]
]

ax = median_length_plot_df.plot(kind="bar", figsize=(10, 5))

plt.title("Median Length Comparison by Class")
plt.xlabel("Class")
plt.ylabel("Median Length")
plt.xticks(rotation=0)
plt.legend(title="Length Metric")
plt.tight_layout()

median_length_figure_path = FIGURES_DIR / "median_length_comparison_by_class.png"
plt.savefig(median_length_figure_path, dpi=300)
plt.show()

print("Median length comparison figure saved to:", median_length_figure_path)
print("File exists:", median_length_figure_path.exists())

print("\nMedian length comparison table:")
display(median_length_comparison)

#Part-14: Final Findings from overall Data-Analysis

In [ ]:
# 1-Create final data analysis findings table

final_data_analysis_findings = pd.DataFrame([
    {
        "finding_area": "dataset_source",
        "finding": "The analysis used the public HuggingFace deepset/prompt-injections dataset stored in the GitHub repository.",
        "evidence": "Raw parquet files loaded successfully from data/raw.",
        "part_2_decision": "Use the same raw GitHub dataset source for data preparation."
    },
    {
        "finding_area": "dataset_size",
        "finding": "The dataset contains 662 prompts in total.",
        "evidence": "Train = 546 prompts; Test = 116 prompts.",
        "part_2_decision": "Keep the official train/test split unchanged."
    },
    {
        "finding_area": "column_structure",
        "finding": "Both train and test datasets contain the expected columns: text and label.",
        "evidence": "Columns = ['text', 'label'].",
        "part_2_decision": "Use text as the model input and label as the binary target."
    },
    {
        "finding_area": "label_meaning",
        "finding": "The dataset uses binary labels: 0 = SAFE and 1 = INJECTION.",
        "evidence": "Only labels [0, 1] were found in both train and test datasets.",
        "part_2_decision": "Keep labels unchanged during data preparation."
    },
    {
        "finding_area": "class_distribution",
        "finding": "The training split has moderate class imbalance, while the test split is nearly balanced.",
        "evidence": "Train: SAFE = 343, INJECTION = 203; Test: SAFE = 56, INJECTION = 60.",
        "part_2_decision": "Use stratified validation splitting and evaluate with Precision, Recall, F1-score, and AUC-ROC."
    },
    {
        "finding_area": "missing_invalid_text",
        "finding": "No missing, empty, whitespace-only, non-string, or symbol-only text issues were found.",
        "evidence": "All missing/invalid text issue counts = 0.",
        "part_2_decision": "No row removal is required for missing or invalid text."
    },
    {
        "finding_area": "duplicates_and_leakage",
        "finding": "No duplicate rows, duplicate text entries, conflicting labels, or train-test leakage were found.",
        "evidence": "Exact and normalised duplicate/leakage checks all returned 0 issues.",
        "part_2_decision": "Keep the official train/test split unchanged."
    },
    {
        "finding_area": "text_length",
        "finding": "INJECTION prompts are substantially longer than SAFE prompts.",
        "evidence": "SAFE median character length = 42; INJECTION median character length = 126. SAFE median word count = 7; INJECTION median word count = 21.",
        "part_2_decision": "Do not remove long prompts automatically; preserve original text."
    },
    {
        "finding_area": "token_length",
        "finding": "Most prompts are below the 512-token transformer limit, but a small number exceed it.",
        "evidence": "Prompts above 512 tokens: DistilBERT = 1, RoBERTa = 2, SecBERT = 1.",
        "part_2_decision": "Use max_length=512 and truncation=True during tokenizer preparation."
    },
    {
        "finding_area": "unusual_characters",
        "finding": "Unusual characters appear in 32 prompts and are mostly associated with INJECTION prompts.",
        "evidence": "Any unusual-character prompts = 32; SAFE = 2, INJECTION = 30.",
        "part_2_decision": "Do not automatically remove unusual characters because formatting may be part of the injection signal."
    },
    {
        "finding_area": "suspicious_patterns",
        "finding": "Command-like and injection-related terms are much more common in INJECTION prompts.",
        "evidence": "Command-like prompts: SAFE = 5, INJECTION = 91. Code-like prompts: SAFE = 0, INJECTION = 14. URL prompts = 0.",
        "part_2_decision": "Use these patterns only as exploratory evidence, not as rule-based classification logic."
    },
    {
        "finding_area": "outliers",
        "finding": "The longest prompts and extreme token-length prompts are INJECTION prompts.",
        "evidence": "Longest 15 prompts are all INJECTION. Two prompts exceed 512 tokens for at least one tokenizer.",
        "part_2_decision": "Keep outliers and handle long prompts using tokenizer truncation."
    },
    {
        "finding_area": "overall_readiness",
        "finding": "The dataset is suitable for Part 2 data preparation and transformer fine-tuning.",
        "evidence": "Dataset passed structural, quality, duplicate, leakage, length, token, unusual-character, pattern, and risk checks.",
        "part_2_decision": "Proceed to data preparation using stratified validation split and transformer tokenization."
    }
])

print("Final data analysis findings table created.")
display(final_data_analysis_findings)

In [ ]:
# 2-Save final data analysis findings outputs

TABLES_DIR = RESULTS_DIR / "tables"
REPORTS_DIR = RESULTS_DIR / "reports"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

final_findings_table_path = TABLES_DIR / "final_data_analysis_findings.csv"
final_report_path = REPORTS_DIR / "final_data_analysis_report.md"

final_data_analysis_findings.to_csv(final_findings_table_path, index=False)

with open(final_report_path, "w", encoding="utf-8") as report_file:
    report_file.write(final_data_analysis_report)

print("Final findings table saved:")
print("final_data_analysis_findings.csv:", final_findings_table_path.exists())

print("\nFinal written report saved:")
print("final_data_analysis_report.md:", final_report_path.exists())